# ECON 524 — Big Data Econometrics
## Computational Assignment 5: DML under Unconfoundedness vs. DML for Difference-in-Differences

**Emory University · Fall 2026**
Instructor: Zheng Fang · Lab instructor: Joel Reyes Mora · Grader: Marcelo Ortiz-Villavicencio

**Time allowed:** ten days from release. **Based on:** Lecture 3 §3 (Difference-in-Differences: Assumptions 3.1–3.2, Theorem 3.1, covariates) and Lecture 4 (orthogonal scores, cross-fitting, the Riesz representer, Automatic DML for DiD). It builds on Assignment 3, Q3 (outcome-regression and IPW DiD) and Assignment 4, Q4(c) (Auto-DML for DiD). Those assignments asked *how* to estimate the ATT under conditional parallel trends. This one asks a different question: **when panel data are available, should you trust selection on observables in levels or parallel trends in changes, and what does each choice cost?**

---

### What this assignment is about
With a pre-period outcome $Y_0$ in hand, an applied researcher can estimate the ATT of a program in two ways, and DML can be used for either:

* **Levels / lagged-dependent-variable (LDV) DML.** Assume treatment is as good as random given the covariates *and* the pre-period outcome, and apply the orthogonal ATT score to $Y_1$ with controls $W=(Z,Y_0)$.
* **DiD-DML.** Assume parallel trends conditional on $Z$, and apply the *same* orthogonal ATT score to the change $\Delta Y = Y_1-Y_0$ with controls $W=Z$.

Both are "double/debiased", so both are robust to regularization bias in the nuisances. Neither is robust to a wrong identifying assumption. Q1 is about **identification**: which assumption fails under which selection mechanism, and why the two estimates bracket the truth. Q2 is about **estimation**: when both assumptions hold, which estimator is more precise, and what pre-period data can and cannot tell you.

### Rules
* **Simulated data only.** The data-generating process (DGP) is fully specified below. Fix and report seeds so that *Restart & Run All* reproduces your numbers.
* **Nuisance learners: LASSO only.** Use cross-validated LASSO for outcome regressions and $\ell_1$-penalized logistic regression for propensity scores (both from `scikit-learn`, with standardized covariates). No trees, forests or neural networks. Implement the DML estimators yourself with `numpy`, `pandas`, `scipy`, `scikit-learn` and `matplotlib`; `doubleml` may be used **only** as an optional cross-check.
* **Every part requires interpretation.** Answer in the markdown cells (typically 4–8 sentences) and refer to slides, formulas, or your tables. Numbers without interpretation earn at most half credit. Derivations marked *by hand* must be typed in Markdown (LaTeX is fine).
* **Runtime.** Everything runs on a laptop CPU. The Monte Carlo in Q2(b) is the most expensive step (roughly 10–20 minutes in total); report its runtime. Monte Carlo sizes are **minimums**.
* Include the **AI-use acknowledgment statement** required by the lab syllabus (README) at the end of the notebook.

**Points:** Q1 50 · Q2 50.

## Notation and the simulation design

**Notation (as in the slides).** Periods $t\in\{-1,0,1\}$; treatment happens only between $t=0$ and $t=1$, so $D\equiv D_1\in\{0,1\}$ and $D_0=D_{-1}=0$. Observed outcomes $Y_t$, potential outcomes $Y_t(0),Y_t(1)$, with no anticipation (Assumption 3.1): $Y_t=Y_t(0)$ for $t\le0$. Covariates $Z\in\mathbb R^p$. The target is
$$\theta_{ATT}=E[Y_1(1)-Y_1(0)\mid D=1].$$
Write $\Delta Y\equiv Y_1-Y_0$, $\pi\equiv P(D=1)$, $\varrho_0(W)\equiv P(D=1\mid W)$ and $\gamma_0(0,W)\equiv E[\,\cdot\mid D=0,W]$ for the control-group regression of the relevant outcome on the controls $W$. $\Sigma$ is the $p\times p$ Toeplitz matrix $\Sigma_{jk}=0.5^{|j-k|}$ and $\operatorname{expit}(u)=1/(1+e^{-u})$.

### DGP-5: a job-training program with a pre-period panel

A training program is offered at $t=1$. We observe earnings (in \$1,000s, centered) in three periods and $p=100$ background covariates.

| Component | Specification |
|---|---|
| Covariates | $Z\sim N(0,\Sigma)$, $p=100$ |
| Unit effect | $\alpha_i = Z_1 + 0.5Z_2 + \sigma_a U_i$, $U_i\sim N(0,1)$ unobserved, $\sigma_a=1$ |
| Transitory shocks | stationary AR(1): $\varepsilon_{-1}\sim N(0,1)$, $\varepsilon_t=\rho\,\varepsilon_{t-1}+\sqrt{1-\rho^2}\,\eta_t$, $\eta_t\sim N(0,1)$ |
| Covariate-specific trend | $\operatorname{tr}(Z)=0.5Z_3+0.5Z_4$ |
| Untreated outcomes | $Y_t(0)=\alpha_i+t\cdot\operatorname{tr}(Z)+\varepsilon_t$, $t\in\{-1,0,1\}$ |
| Treatment | $P(D=1\mid Z,U,Y_0)=\operatorname{expit}\{-0.5+0.5Z_3+0.5Z_5+c_U\,U+c_Y\,(Y_0-Z_1-0.5Z_2)\}$ |
| Treatment effect | $Y_1=Y_1(0)+D\,\tau(Z)$, $\tau(Z)=1+0.5Z_1$ |

Default $\rho=0.5$. Three **selection scenarios**:

| Scenario | $c_U$ | $c_Y$ | Story |
|---|---|---|---|
| **A** — selection on a permanent unobservable | $-0.75$ | $0$ | Workers with low unobserved ability $U$ are more likely to enrol. |
| **B** — selection on the pre-period outcome (Ashenfelter's dip) | $0$ | $-0.75$ | Caseworkers enrol workers whose earnings at $t=0$ were low *relative to what their covariates predict*. |
| **C** — selection on observables only | $0$ | $0$ | Enrolment depends on $Z$ only. |

Note that $Z_3$ enters both the trend and the propensity, so **unconditional** parallel trends fails in every scenario.

## Setup

*Imports, seeds, your implementation of DGP-5 and of the DML estimator go here.*

In [ ]:
# Setup: imports, seeds, DGP-5, DML estimator


---
## Question 1 — Two identifying assumptions, one score (50 pts)

*Lecture 3 slides 36–38 and §3 with covariates; Lecture 4 slides 29–32 and 49–50.*

The two candidate assumptions are

* **(LDV) Unconfoundedness given the lagged outcome:** $\;Y_1(0)\perp D\mid W$, with $W=(Z,Y_0)$, and $\varrho_0(W)<1$.
* **(CPT) Conditional parallel trends:** $\;E[Y_1(0)-Y_0(0)\mid D=1,Z]=E[Y_1(0)-Y_0(0)\mid D=0,Z]$, with $\varrho_0(Z)<1$.

### Q1(a) [15 pts] Identification and the orthogonal score, *by hand*

1. Show that under (LDV), $\theta_{ATT}=E\big[Y_1-\gamma_0(0,W)\mid D=1\big]$ with $\gamma_0(0,W)=E[Y_1\mid D=0,W]$, and that under (CPT) and no anticipation, $\theta_{ATT}=E\big[\Delta Y-\gamma_0(0,Z)\mid D=1\big]$ with $\gamma_0(0,Z)=E[\Delta Y\mid D=0,Z]$. Conclude that both are the same functional, $E[\tilde Y-\gamma_0(0,W)\mid D=1]$, applied to different pairs $(\tilde Y,W)$.
2. For this functional, the orthogonal (doubly robust) moment is
$$E\Big[\underbrace{\Big(D-\frac{(1-D)\,\varrho_0(W)}{1-\varrho_0(W)}\Big)}_{\pi\cdot\alpha_0(D,W)\ \text{(Riesz representer, up to }\pi)}\big\{\tilde Y-\gamma_0(0,W)\big\}-D\,\theta_{ATT}\Big]=0 .$$
Verify that it holds at the truth, and show that its Gateaux derivatives with respect to $\gamma(0,\cdot)$ and $\varrho(\cdot)$ are zero (Neyman orthogonality). Relate the weight on control units to the DiD Riesz representer on Lecture 4 slide 50.
3. Suppose the control regression is linear, $\gamma(0,Z,Y_0)=\beta Y_0+Z^\top\delta$. Show that the LDV score with $\beta=1$ is *identical* to the DiD score with $\gamma(0,Z)=Z^\top\delta$. In one sentence: what restriction does DiD impose that the LDV approach estimates?

**Q1(a): your answer / interpretation.**

_Write here._

### Q1(b) [20 pts] Which assumption holds in which scenario?

1. *By hand.* For each of Scenarios A, B and C, decide whether (LDV) and/or (CPT) holds in DGP-5. Justify each answer from the DGP. For the two failures, sign the bias of the corresponding estimator. *Hint for Scenario B:* conditional on $Z$, write $S\equiv \sigma_aU+\varepsilon_0$ and show that $E[\varepsilon_1-\varepsilon_0\mid Z,S]=\kappa S$ with $\kappa=(\rho-1)/(\sigma_a^2+1)$.
2. Compute $\theta_{ATT}$ and $\pi$ for each scenario from a population of $2\times10^6$ draws.
3. On one sample with $n=4{,}000$ for each scenario, compute: (i) the unconditional DiD (difference in mean $\Delta Y$ between groups); (ii) LDV-DML; (iii) DiD-DML. Use 5-fold cross-fitting; report point estimates, influence-function standard errors that account for estimating $\pi$, and 95% confidence intervals. Clip estimated propensities to $[0.01,0.99]$ and say so.
4. Interpret: does every estimator behave as your answer to part 1 predicts? Why does being "double/debiased" not help a DML estimator whose identifying assumption is wrong?

In [ ]:
# Q1(b): your code here


**Q1(b): your answer / interpretation.**

_Write here._

### Q1(c) [15 pts] Bracketing

Real selection is rarely pure A or pure B. Let $w\in\{0,0.25,0.5,0.75,1\}$ and set $c_U=-0.75(1-w)$, $c_Y=-0.75w$, so that $w=0$ is Scenario A and $w=1$ is Scenario B.

1. For each $w$, compute $\theta_{ATT}$ (population) and the LDV-DML and DiD-DML estimates on one sample with $n=4{,}000$. Plot the three against $w$, with 95% intervals for the two estimates.
2. Interpret the picture. Relate it to the bracketing result in Angrist and Pischke (2009, §5.4) and Ding and Li (2019): if selection is on a permanent unobservable *or* on the past outcome, the two estimates bracket the ATT. Explain in words why the biases go in **opposite directions** in this DGP (hint: what does each estimator do with the mean reversion of $\varepsilon$?).
3. Ding and Li prove bracketing under specific conditions. Name one feature of DGP-5 that their argument relies on, and describe a change to DGP-5 that you would expect to break bracketing. You do not need to simulate it.

In [ ]:
# Q1(c): your code here


**Q1(c): your answer / interpretation.**

_Write here._

---
## Question 2 — When both are valid: efficiency, inference and what the data can tell you (50 pts)

*Lecture 4 slides 14–18 (orthogonality and the asymptotic variance of DML), slides 49–50; Lecture 3 placebo/pre-trend discussion.*

Throughout Q2(a)–(b) use **Scenario C**, where (LDV) and (CPT) both hold.

### Q2(a) [15 pts] Which estimator is more precise? *By hand*

1. Under Scenario C, let $r_1\equiv Y_1(0)-E[Y_1(0)\mid Z]$ and $r_0\equiv Y_0-E[Y_0\mid Z]$. Using only the DGP, compute $\operatorname{Var}(r_0)$, $\operatorname{Var}(r_1)$ and $\operatorname{Cov}(r_1,r_0)$ as functions of $(\sigma_a^2,\rho)$.
2. The residual of the LDV control regression is $r_1-\beta^\ast r_0$ with $\beta^\ast=\operatorname{Cov}(r_1,r_0)/\operatorname{Var}(r_0)$; the residual of the DiD control regression is $r_1-r_0$. Show that
$$V_{DiD}-V_{LDV}\;\equiv\;\operatorname{Var}(r_1-r_0)-\operatorname{Var}(r_1-\beta^\ast r_0)=\frac{(1-\rho)^2}{\sigma_a^2+1}\;\ge 0 .$$
3. These residual variances enter the variance of the orthogonal score, $E[\psi^2]$, through the control-weight term. Explain why this makes LDV-DML at least as precise as DiD-DML when both are valid, and when the gap vanishes. Connect your answer to McKenzie (2012) on ANCOVA vs. DiD in experiments.

**Q2(a): your answer / interpretation.**

_Write here._

### Q2(b) [20 pts] Monte Carlo: bias, precision and coverage

1. Run $R\ge200$ replications of Scenario C with $n=1{,}000$, $p=100$, for each $\rho\in\{0,0.5,0.9\}$. In each replication compute LDV-DML and DiD-DML (5-fold cross-fitting, LASSO nuisances) with their influence-function standard errors.
2. For each estimator and $\rho$, report the bias, the Monte Carlo SD, the mean estimated standard error and the coverage of the 95% interval. Add the ratio $\mathrm{SD}_{DiD}/\mathrm{SD}_{LDV}$ and the theoretical ratio $\sqrt{V_{DiD}/V_{LDV}}$ from Q2(a). Report the runtime.
3. Interpret: are both estimators approximately unbiased with close to nominal coverage, as DML theory predicts for a correctly specified identifying assumption? Does the efficiency ranking match Q2(a)? Why is the empirical SD ratio not exactly equal to $\sqrt{V_{DiD}/V_{LDV}}$? (Hint: what other terms does $E[\psi^2]$ contain?) Taken together with Q1, describe the trade-off a researcher faces.

In [ ]:
# Q2(b): your code here


**Q2(b): your answer / interpretation.**

_Write here._

### Q2(c) [15 pts] What can the pre-period tell you? Placebo tests

The extra pre-period $Y_{-1}$ allows a placebo test: pretend treatment happened between $t=-1$ and $t=0$, where the true effect is zero.

1. On the Scenario A and Scenario B samples from Q1(b) (and Scenario C for reference), compute (i) the **DiD placebo**: DiD-DML with outcome $Y_0-Y_{-1}$ and controls $Z$; (ii) the **LDV placebo**: LDV-DML with outcome $Y_0$ and controls $(Z,Y_{-1})$. Report estimates, standard errors and a test of zero.
2. Which placebo rejects in which scenario? Explain each result from the DGP. In particular, explain why the LDV placebo rejects in Scenario B even though LDV-DML is *valid* for the real effect in Scenario B.
3. Write a short recommendation (one paragraph) for an applied researcher who has two pre-periods and a high-dimensional $Z$: what would you estimate and report, what does a passed DiD pre-trend test certify, and what does it *not* certify?

In [ ]:
# Q2(c): your code here


**Q2(c): your answer / interpretation.**

_Write here._

---
## References

- Angrist, J. D., and Pischke, J.-S. (2009). *Mostly Harmless Econometrics*, §5.4. Princeton University Press.
- Ashenfelter, O. (1978). Estimating the effect of training programs on earnings. *Review of Economics and Statistics*, 60(1), 47–57.
- Chernozhukov, V., Chetverikov, D., Demirer, M., Duflo, E., Hansen, C., Newey, W., and Robins, J. (2018). Double/debiased machine learning for treatment and structural parameters. *The Econometrics Journal*, 21, C1–C68.
- Chang, N.-C. (2020). Double/debiased machine learning for difference-in-differences models. *The Econometrics Journal*, 23(2), 177–191.
- Ding, P., and Li, F. (2019). A bracketing relationship between difference-in-differences and lagged-dependent-variable adjustment. *Political Analysis*, 27(4), 605–615.
- McKenzie, D. (2012). Beyond baseline and follow-up: The case for more T in experiments. *Journal of Development Economics*, 99(2), 210–221.
- Sant'Anna, P. H. C., and Zhao, J. (2020). Doubly robust difference-in-differences estimators. *Journal of Econometrics*, 219(1), 101–122.

---
## AI-use acknowledgment
*Paste the acknowledgment statement required by the lab syllabus here, describing which tools you used, for which parts, and how you verified their output.*